# TPM034A Machine Learning for socio-technical systems 
## `Lab session 03: Predicting visual attractiveness with semantic segmentation`

**Delft University of Technology**<br>
**Q2 2026**<br>
**Instructor:** Sander van Cranenburgh & Francisco Garrido Valenzuela<br>

### `Instructions`

**Lab sessions aim to:**<br>
* Show and reinforce how models and ideas presented in class are used in practice.<br>
* Help you gather hands-on machine learning skills.<br>

**Lab sessions are:**<br>
* Learning environments where you work with Jupyter notebooks and where you can get support from TAs and fellow students.<br> 
* Not graded and do not have to be submitted.

### `Use of AI tools`

AI tools, such as ChatGPT and GitHub Copilot, are powerful tools to assist with programming. Moreover, in your future careers, you will work in a world where such tools are widely available. As such, we encourage you to learn to use AI tools effectively.

However, remember that the **primary objective of lab sessions is not to finish it, but to learn from it and develop your skills**. AI tools can help you achieve this, but they can also undermine your learning if you rely on them too heavily, see this recent [article](../../Assets/chatbots-and-thinking-english.pdf). Being able to instruct an AI tool to solve a problem is not the same as understanding how to solve it yourself. Understanding requires some effort and friction. You should therefore remain actively involved in conceptualising problems, structuring analyses, interpreting results, and critically evaluating AI-generated solutions.

> ⚠️ **Important:** We recommend using AI tools carefully, particularly because you will **NOT** have access to them during the *practical exam* (as internet access will be restricted). Therefore, at all times, make sure you understand what you are doing, not just that your code works.

### `Google Colab workspace set-up`

Uncomment the following cells code lines if you are running this notebook on Colab

In [ ]:
#!git clone https://github.com/TPM034A/TPM034A_2026
#!pip install -r TPM034A_2026/requirements.txt
#!mv "/content/TPM034A_2026/Lab_sessions/lab_session_03/data" /content/data

### `Application: Predicting perceived visual neighbourhood attractiveness with semantic urban features` <br>

In this lab session, we explore how computer vision and machine learning can be combined to understand the **perceived visual attractiveness (livability) of neighbourhoods** in Rotterdam. 

Understanding what makes an urban environment visually attractive is essential for socio-technical decision-making. Municipalities need to identify which neighbourhoods require revitalization, urban planners aim to optimise the allocation of green spaces, and researchers study how street aesthetics influence residential location choice, public health, and house prices.

In this lab session, we adopt an **explainable socio-technical approach**: we use **Semantic Segmentation**, specifically, a state-of-the-art **Mask2Former (Swin-Large)** vision model (see [this paper](https://arxiv.org/abs/2112.01527)) trained on the [**ADE20K** dataset](https://doi.org/10.1007/s11263-018-1140-0) , which classifies every pixel of a street-level image into one of **150 semantic categories** (e.g. buildings, trees, sky, roads, water, cars, grass, fences, and many more).

By computing the pixel proportion (%) of each category, we transform raw images into **tangible, policy-relevant urban metrics**. However, many of the 150 ADE20K categories are not relevant for outdoor urban scenes (e.g. bed, sofa, oven). A key task in this lab is to **select the subset of categories** most appropriate for urban analysis. This is a practical exercise in feature engineering and domain-driven decision-making.

`In this lab session, we aim to develop and compare multiple ML models (Linear Regression, Random Forest, and Multilayer Perceptrons) that map semantic urban composition to visual attractiveness levels, and critically reflect on the socio-technical interpretability of our models.`

**Where do the true labels come from?**<br>

Since we use supervised learning, we need to have data containing the true labels. But where do the true labels for visual attractiveness come from? The true labels come from a computer vision model that is trained on data from a so-called discrete choice experiment. In this experiment, people were placed in the hypothetical situation that they had to move to a different neighbourhood and were given two options. The visual attractiveness is learned from their choices. 
 
See the figure below for an example of one such choice task. More information about the survey and the model can be found in [this paper](https://www.sciencedirect.com/science/article/pii/S0965856424003483).

<div>
<center><img src="assets/survey_snapshot.png" width="500"/></center>
</div>

**Why do we need another model to (re)do the mapping?**<br>

The mapping from image to visual attractiveness indeed already comes from a model. Therefore, one may wonder why do we need another model? Applying complex deep learning computer vision models directly to raw images is computationally expensive and requires dedicated GPU infrastructure. Moreover, image embeddings, interal representation of the images inside the Computer vision model are uninterpretable for city planners. 

Having a fast, interpretable proxy model based on semantic percentages allows researchers and municipalities to easily simulate interventions (e.g. *What happens to perceived attractiveness if we convert 15% of asphalt road into roadside vegetation?*). For instance, building a city-wide map about visual attractiveness like the one below for Delft requires processing over 400,000 images.

<div>
<center><img src="assets/visual_attractiveness_Delft.png" width="500"/></center>
</div>

**Data for this lab**<br>

In this lab session you will work with four datasets:

1. `data/image_tabular/image_metadata.csv`: A table CSV file with image metadata (e.g., year, month, latitude, longitude, hexagon ID, and visual attractiveness score) of Rotterdam street view images.
2. `data/image_tabular/image_segmentation.csv`: A table CSV file containing the pixel proportions (%) of **150 ADE20K semantic categories** for each image, extracted using a Mask2Former (Swin-Large) model.
3. `data/geo/hexagons.gpkg`: A geospatial dataset of Rotterdam partitioned into spatial hexagons.
4. `data/images`: A folder with sample street view images from Rotterdam.

The first three tabular and geospatial files are already located in the `data` folder. The `images` sample folder can be downloaded automatically using the script below. We use a representative subset of 1,000 images (around 140 MB) for visual inspections.

In [ ]:
## IMPORTANT: You have to be on the TUDelft network (eduroam) or under eduVPN to run this script
download_full_dataset = False

from assets import image_downloader as imd
imd.download_images(download_full_dataset)

**Learning objectives**. After completing the following exercises you will be able to: <br>
1. Work with **semantic segmentation** data and understand how pixel-level classifications translate into urban morphology metrics.
2. Apply **feature engineering** by selecting relevant urban classes from a large-scale (150-class) semantic vocabulary.
3. Formulate and verify domain hypotheses connecting urban physical features (e.g., tree canopy, building density, water bodies) to perceived attractiveness.
4. Train, tune, and evaluate multiple ML models, including `Multiple Linear Regression`, `Random Forest`, and `Multi-Layer Perceptron (MLP)`.
5. Contrast the interpretability of linear models and tree-based feature importances against black-box representations.
6. Critically reflect on the socio-technical limitations of visual-only predictive models for public policy and municipal planning.

#### `Organisation`
This lab session comprises **`3`** parts:

1. **Loading and exploring the urban data**<br>
    1.1. Reading the metadata and geospatial data<br>
    1.2. Exploring the image metadata and target distribution<br>
    1.3. Exploring the geospatial distribution of attractiveness<br>
    1.4. Visual inspection of images and attractiveness levels<br>

2. **Semantic Segmentation: From Pixels to Urban Features**<br>
    2.1. Understanding the ADE20K segmentation model (150 classes)<br>
    2.2. Feature engineering: selecting relevant urban classes<br>
    2.3. Exploring distributions and correlations of the selected features<br>

3. **Training multiple models: Regression model, Random forest and Multilayer perceptron (MLP)**<br>
    3.1. Preparing the dataset (standardisation and train/test split)<br>
    3.2. Linear multiple regression model (baseline & coefficients)<br>
    3.3. Random Forest (hyperparameter tuning & feature importances)<br>
    3.4. Multilayer Perceptron (MLP) & GridSearch optimization<br>
    3.5. Model comparison, prediction behavior, and socio-technical reflections<br>

In [ ]:
# Basic libraries
import numpy as np
import pandas as pd
import geopandas as gpd
import warnings

# Suppress deprecation and unpickling version warnings for a clean educational notebook
warnings.filterwarnings('ignore', category=FutureWarning)
try:
    from sklearn.exceptions import InconsistentVersionWarning
    warnings.filterwarnings('ignore', category=InconsistentVersionWarning)
except ImportError:
    pass

# ML tools
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, make_scorer

# Models
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor

# Visualization libraries
import seaborn as sns
from matplotlib import pyplot as plt
from matplotlib.ticker import FixedLocator
from branca.element import Figure

# Other libraries
from pathlib import Path
from shapely.geometry import Point
from PIL import Image
import pickle

#### **1. Loading and exploring the data**
##### 1.1. Reading the metadata and geospatial data

Before constructing predictive models, we explore our datasets. First, we load the image metadata (*data/image_tabular/image_metadata.csv*) which contains geographic and temporal information along with the target attractiveness scores, and the geospatial dataset (*data/geo/hexagons.gpkg*) which defines the spatial analysis units in Rotterdam.

In [ ]:
# Data folder path
data = Path('data')

# Read datasets
img_metadata = pd.read_csv(data/'image_tabular'/'image_metadata.csv')
hexagons = gpd.read_file(data/'geo'/'hexagons.gpkg')

In [ ]:
# Verify geographic coordinate system / projection
print(f"Hexagons CRS: {hexagons.crs}")

##### 1.2. Exploring the image metadata

The `img_metadata` DataFrame contains metadata for street view images in Rotterdam. Let's inspect the first rows:

In [ ]:
img_metadata.head(5)

The dataset contains:
* `img_id`: Unique identifier of the image.
* `img_path`: File name of the image.
* `year` & `month`: Date when the street-level photograph was captured.
* `lat` & `lng`: Geographic coordinates (WGS84).
* `hex_id`: Spatial hexagon unit where the image is situated.
* `attractiveness`: Visual attractiveness score estimated from the discrete choice experiment.
* `in_folder`: Binary flag indicating whether the image is included in the 1,000-image sample folder.

Let's look at the summary statistics of `img_metadata`:

In [ ]:
img_metadata.describe()

In [ ]:
# Histogram of the attractiveness at image level
fig, ax = plt.subplots(figsize=(7, 3.5))
sns.histplot(img_metadata['attractiveness'], kde=True, ax=ax, color='steelblue')
ax.set_title('Distribution of Perceived Visual Attractiveness')
ax.set_xlabel('Attractiveness Score')
ax.set_ylabel('Count')
plt.show()

Notice that the visual attractiveness score is roughly bell-shaped, centered near 0 with standard deviation around 1. Most scores range between -3 and +3.

##### 1.3. Exploring the geospatial distribution of attractiveness

To understand spatial patterns across Rotterdam, we can aggregate image scores to the hexagon level. We define a helper function to convert tabular coordinates to the Dutch spatial reference system (Amersfoort / RD New, EPSG:28992):

In [ ]:
def dataframe_to_geodataframe_nl(original_dataframe, latitud_column_name, longitude_column_name):
    '''
    Converts a DataFrame into a GeoDataFrame using latitude and longitude columns (EPSG:4326),
    and transforms the CRS to EPSG:28992 (Rijksdriehoekstelsel, the projection used in the Netherlands).
    '''
    original_dataframe['geometry'] = original_dataframe.apply(
        lambda x: Point(x[longitude_column_name], x[latitud_column_name]), axis=1
    )
    geodataframe = gpd.GeoDataFrame(original_dataframe, geometry='geometry', crs=4326)
    return geodataframe.to_crs(28992)

### <span style="color:skyblue">Exercise 1: Explore the geospatial distribution of attractiveness</span> 
`A` Compute the average attractiveness *per hexagon* (`hex_id`) using `.groupby()`.<br>
`B` Merge these average attractiveness scores with the `hexagons` GeoDataFrame on the column `hex_id`.<br>
`C` Visualise the average attractiveness of each neighbourhood using the `.explore()` method of GeoPandas.<br>
`D` Do you observe distinct spatial patterns in Rotterdam? (Hint: compare green suburban areas, parks, waterways, city center, and industrial/harbor zones).

In [ ]:
# ADD HERE YOUR ANSWER TO EXERCISE 1



##### 1.4. Visual inspection of images and attractiveness

Let's inspect sample images to understand visually what high and low attractiveness scores correspond to.

In [ ]:
# Visual inspection of sample images
visual_images = img_metadata.loc[img_metadata['in_folder'] == 1].copy()

# Sample and display one image
rnd_img = visual_images.sample(1, random_state=42).iloc[0]
img_file_path = data/'images'/rnd_img['img_path']

if img_file_path.exists():
    fig, ax = plt.subplots(figsize=(6, 3.5))
    im = Image.open(img_file_path)
    ax.imshow(im)
    ax.set_title(f"Image: {rnd_img['img_path']} | Attractiveness: {rnd_img['attractiveness']:.2f}")
    ax.axis('off')
    plt.show()
else:
    print(f"Sample image {rnd_img['img_path']} not found locally. Run image downloader above.")

We can also inspect the extreme ends of the distribution: what do the **least attractive** vs. the **most attractive** street scenes in Rotterdam look like?

In [ ]:
# Display top 3 and bottom 3 attractive images
lowest_attr = visual_images.sort_values(by='attractiveness', ascending=True).head(3)
highest_attr = visual_images.sort_values(by='attractiveness', ascending=False).head(3)

fig, axes = plt.subplots(2, 3, figsize=(15, 6))

for i, (_, row) in enumerate(lowest_attr.iterrows()):
    p = data/'images'/row['img_path']
    if p.exists():
        axes[0, i].imshow(Image.open(p))
    axes[0, i].set_title(f"Low: {row['attractiveness']:.2f}")
    axes[0, i].axis('off')

for i, (_, row) in enumerate(highest_attr.iterrows()):
    p = data/'images'/row['img_path']
    if p.exists():
        axes[1, i].imshow(Image.open(p))
    axes[1, i].set_title(f"High: {row['attractiveness']:.2f}")
    axes[1, i].axis('off')

axes[0, 0].set_ylabel("Lowest Attractiveness", fontsize=12, labelpad=20)
axes[1, 0].set_ylabel("Highest Attractiveness", fontsize=12, labelpad=20)
plt.tight_layout()
plt.show()

**Visual observation:**
Notice the striking physical differences between the top and bottom rows:
* **High attractiveness:** Characterized by mature trees, lush greenery, open skies, pedestrian walkways, and well-maintained architecture.
* **Low attractiveness:** Dominated by roads, concrete barrier walls, brick infrastructure.

This visual intuition provides the exact motivation for **Semantic Segmentation**!

#### **2. Semantic Segmentation: From Pixels to Urban Features**
##### 2.1. Understanding the segmentation model

Until now, we have visually inspected the images. But how do we quantitatively feed street images into machine learning models?

Instead of relying on abstract latent vectors (such as CNN feature embeddings where individual dimensions lack human meaning), we use **Semantic Segmentation**. Semantic segmentation is a computer vision task where a neural network assigns a categorical class label to **every single pixel** in an image.

In this lab, the images have been pre-processed using **Mask2Former (Swin-Large)**, a state-of-the-art architecture trained on the **ADE20K** dataset. ADE20K is a large-scale scene understanding benchmark that covers **150 semantic categories** — from common urban elements (buildings, trees, roads, sky) to indoor objects (beds, sofas, ovens) and rare classes (arcade machines, pool tables).

For each image, the surface share of each category is calculated as:
$$\text{Class Percentage (\%)} = \frac{\text{Number of pixels belonging to class } c}{\text{Total pixels in image}} \times 100$$

Let's load the pre-extracted segmentation dataset:

In [ ]:
# Load the ADE20K semantic segmentation dataset (150 classes)
segmentation = pd.read_csv(data/'image_tabular'/'image_segmentation.csv')
print(f"Shape: {segmentation.shape} ({segmentation.shape[1] - 1} semantic classes + img_id)")
segmentation.head(3)

As you can see, there are **150 columns** representing the pixel coverage (%) for each ADE20K class. However, many of these classes are not relevant for outdoor urban street-level analysis. Let's explore which categories are actually present in our Rotterdam images:

In [ ]:
# Average pixel coverage per class across all images
class_means = segmentation.drop(columns=['img_id']).mean().sort_values(ascending=False)

# How many classes have meaningful presence?
print(f"Classes with avg > 1%:   {(class_means > 1).sum()}")
print(f"Classes with avg > 0.1%: {(class_means > 0.1).sum()}")
print(f"Classes with avg < 0.01%: {(class_means < 0.01).sum()}")

# Bar chart: Top 25 most present classes
fig, ax = plt.subplots(figsize=(12, 4.5))
top_classes = class_means.head(25)
sns.barplot(x=top_classes.index, y=top_classes.values, ax=ax, hue=top_classes.index, palette='mako', legend=False)
ax.set_title('Top 25 ADE20K Classes by Average Pixel Coverage in Rotterdam SVIs')
ax.set_ylabel('Mean Area (%)')
ax.set_xlabel('Semantic Class')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

Notice that:
* Street scenes in Rotterdam are dominated by a handful of classes: `building`, `sky`, `tree`, `road`, `grass`, `sidewalk`, `car`, and `plant`.
* Over **100 classes** have an average coverage below 0.01% — they are practically absent from outdoor urban imagery.
* Many ADE20K categories are clearly **indoor** (e.g. `bed`, `sofa`, `bathtub`, `oven`, `refrigerator`) or irrelevant for street-level urban analysis (e.g. `pool table`, `arcade machine`).

**This raises an important question in applied machine learning:** Should we use all 150 features as inputs to our models, or should we apply **domain knowledge** to select a relevant subset?

##### 2.2. Feature engineering: selecting relevant urban classes

In many real-world ML applications, the raw feature space contains noise, irrelevant variables, or redundancies. **Feature selection** — the process of identifying and retaining only the most informative and domain-appropriate variables — is a critical step that can improve model interpretability, reduce overfitting, and make results actionable for stakeholders (in our case, urban planners).

For this lab, you need to examine the 150 ADE20K classes and select a subset of **outdoor urban classes** that you believe are relevant for predicting neighbourhood attractiveness.

### <span style="color:skyblue">Exercise 2: Feature selection — choosing relevant urban classes</span>
`A` Examine the 150 ADE20K classes. How many have a mean pixel coverage above 0.1%? List the top 20 classes by average presence.<br>
`B` Using your domain knowledge about urban environments, select a subset of approximately **12 to 18 classes** that you consider relevant for outdoor urban attractiveness prediction. Group them into intuitive categories (e.g. vegetation, water, built environment, infrastructure, vehicles).<br>
`C` Justify why you excluded specific high-presence classes (if any) and why you included low-presence classes (if any).<br>
`D` Create a filtered DataFrame `seg_selected` containing only `img_id` and your selected columns.

In [ ]:
# ADD HERE YOUR ANSWER TO EXERCISE 2



##### 2.3. Exploring the selected features

Now that we have our curated feature set, let's explore the distributions and their relationship with attractiveness.

In [ ]:
# Summary statistics of selected features (this variable you should define it)
seg_selected.describe()

In [ ]:
# Average presence of selected classes
selected_means = seg_selected.drop(columns=['img_id']).mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 4))
sns.barplot(x=selected_means.index, y=selected_means.values, ax=ax, hue=selected_means.index, palette='mako', legend=False)
ax.set_title('Average Pixel Coverage: Selected Urban Classes')
ax.set_ylabel('Mean Area (%)')
ax.set_xlabel('Semantic Class')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### <span style="color:skyblue">Exercise 3: Analyzing correlations and formulating urban planning hypotheses</span>
`A` Merge the selected segmentation features with the image metadata (`attractiveness`) using `img_id`.<br>
`B` Compute and display the correlation matrix (heatmap) between the selected semantic classes and visual `attractiveness`.<br>
`C` Which 3 urban elements have the strongest **positive** correlation with attractiveness? Which have the strongest **negative** correlation?<br>
`D` **Policy Hypothesis:** Based on these correlations, if the Rotterdam city council wants to improve perceived neighbourhood attractiveness, what physical interventions does this data suggest?

In [ ]:
# ADD HERE YOUR ANSWER TO EXERCISE 3



#### **3. Training multiple models: Regression model, Random forest and Multilayer perceptron (MLP) for predicting attractiveness**<br>

In machine learning for socio-technical systems, researchers rarely rely on a single model. Different algorithm families possess distinct inductive biases:
1. **Multiple Linear Regression (Benchmark):** Highly interpretable, assumes linear additive relationships.
2. **Random Forest (RF):** Non-linear ensemble model, captures complex feature thresholds and interactions without parametric assumptions.
3. **Multilayer Perceptron (MLP):** Universal function approximator, capable of learning smooth non-linear manifolds.

Comparing these models allows us to evaluate whether non-linear complexity provides tangible predictive gains over a transparent linear baseline.

##### 3.1. Preparing the dataset

We merge the selected segmentation features with the target variable `attractiveness`, standardize the features, and split the data into training (80%) and testing (20%) sets.

In [ ]:
# Merging selected segmentation with metadata
df = pd.merge(img_metadata[['img_id', 'attractiveness']], seg_selected, on='img_id').copy()

# Define features and target
feature_cols = selected_classes
predict = 'attractiveness'

# Standardize features and target
scaler = StandardScaler()
df[feature_cols + [predict]] = scaler.fit_transform(df[feature_cols + [predict]])

# Split dataset into train and test sets
X_train, X_test, Y_train, Y_test = train_test_split(
    df[feature_cols], df[predict], test_size=0.2, random_state=44
)

print(f"Training samples: {len(X_train)} | Test samples: {len(X_test)}")
print(f"Features used: {len(feature_cols)} → {feature_cols}")

We define a standardized performance evaluation function computing **Mean Squared Error (MSE)**, **Mean Absolute Error (MAE)**, and **R² Score**:

In [ ]:
def eval_regression_perf(model, X_train, X_test, Y_train, Y_test):
    '''
    Evaluates regression model performance on train and test sets.
    '''
    Y_pred_train = model.predict(X_train)
    Y_pred_test = model.predict(X_test)

    def perfs(Y, Y_pred):
        mse = mean_squared_error(Y, Y_pred)
        mae = mean_absolute_error(Y, Y_pred)
        r2 = r2_score(Y, Y_pred)
        return mse, mae, r2

    mse_train, mae_train, r2_train = perfs(Y_train, Y_pred_train)
    mse_test, mae_test, r2_test = perfs(Y_test, Y_pred_test)
        
    print('Performance')
    print(f'Mean Squared  Error Train | Test: \t{mse_train:>7.4f}\t|  {mse_test:>7.4f}')
    print(f'Mean Absolute Error Train | Test: \t{mae_train:>7.4f}\t|  {mae_test:>7.4f}')
    print(f'R2                  Train | Test: \t{r2_train:>7.4f}\t|  {r2_test:>7.4f}\n')

##### 3.2. Linear multiple regression model

We train an ordinary least squares (OLS) multiple linear regression model as our baseline:

In [ ]:
# Create and train multiple linear regression
regr = LinearRegression(fit_intercept=True)
regr.fit(X_train, Y_train)

# Evaluate performance
eval_regression_perf(regr, X_train, X_test, Y_train, Y_test)

# Extract and visualize regression coefficients
regr_coefs = pd.DataFrame(index=feature_cols, data=regr.coef_, columns=['Coefficient'])
regr_coefs.sort_values(by='Coefficient', ascending=False).plot(kind='bar', figsize=(10, 4), color='steelblue')
plt.title('Standardized Linear Regression Coefficients')
plt.ylabel('Coefficient Value')
plt.axhline(0, color='gray', linestyle='--')
plt.tight_layout()
plt.show()

### <span style="color:skyblue">Exercise 4: Interpreting the Linear Baseline</span>
`A` Inspect the regression coefficients. Which urban features contribute most positively to predicted attractiveness, and which have the strongest negative coefficients?<br>
`B` Does any coefficient sign surprise you when compared to the simple correlations from Exercise 3? (Hint: consider multicollinearity between urban classes like roads, sidewalks, and buildings).<br>
`C` What is the key socio-technical advantage of having named semantic features in this linear model compared to training the same regression on abstract CNN embeddings (`feat_01`... `feat_64`)?

In [ ]:
# ADD HERE YOUR ANSWER TO EXERCISE 4

##### 3.3. Random Forest (RF)

Next, we evaluate a non-linear ensemble model: `RandomForestRegressor`. We tune the maximum tree depth (`max_depth`) and the maximum leaf nodes (`max_leaf_nodes`) via 5-fold cross-validation.

In [ ]:
# Create RF regressor
rf_gs = RandomForestRegressor(n_estimators=100, criterion="squared_error", random_state=5)

# Hyperparameter search space
hyperparameter_space = {
    'max_depth': [5, 10, 20],
    'max_leaf_nodes': [20, 50, 100]
}

scorer = make_scorer(mean_squared_error, greater_is_better=False)
rf_gridsearch = GridSearchCV(rf_gs, hyperparameter_space, n_jobs=-1, cv=5, scoring=scorer, return_train_score=True)

In [ ]:
# Train or load cached gridsearch
pickle_file = Path('RF_gridsearch_v2.pickle')

if not pickle_file.exists():
    print("Fitting Random Forest GridSearch (this may take 1-2 minutes)...")
    rf_gridsearch.fit(X_train, Y_train)
    with open(pickle_file, 'wb') as f:
        pickle.dump(rf_gridsearch, f)
else:
    print("Loading pre-trained Random Forest GridSearch...")
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        with open(pickle_file, 'rb') as f:
            rf_gridsearch = pickle.load(f)

print(f'Best Parameters: {rf_gridsearch.best_params_}')
print(f'Best CV MSE: {(-rf_gridsearch.best_score_):.4f}')

In [ ]:
# Train optimal Random Forest model
best_depth = rf_gridsearch.best_params_['max_depth']
best_leaves = rf_gridsearch.best_params_['max_leaf_nodes']

rf = RandomForestRegressor(
    n_estimators=100, max_depth=best_depth, max_leaf_nodes=best_leaves,
    random_state=42, n_jobs=-1
)
rf.fit(X_train, Y_train)

# Evaluate Random Forest
eval_regression_perf(rf, X_train, X_test, Y_train, Y_test)

Let's examine which features are found to be most important by the Random Forest model:

In [ ]:
# Plot Random Forest Feature Importances
feat_importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 6))
sns.barplot(x=feat_importances.values, y=feat_importances.index, ax=ax, hue=feat_importances.index, palette='viridis', legend=False)
ax.set_title('Random Forest Feature Importances (Gini Impurity Reduction)')
ax.set_xlabel('Relative Importance')
plt.tight_layout()
plt.show()

print("Top 5 Most Important Features in Random Forest:")
print(feat_importances.head(5))

Notice how `tree`, `building`, and `grass` dominate the tree splits, together accounting for a substantial majority of the predictive power of the Random Forest.

#### 3.4 Multilayer Perceptron (MLP)

Now we apply a neural network: `MLPRegressor`. Neural networks can approximate complex multi-dimensional non-linear functions.

### <span style="color:skyblue">Exercise 5: Apply GridSearch now using MLP to predict the attractiveness</span>
`A` Define an MLP Regressor and perform a grid search over:
* `hidden_layer_sizes`: e.g. `[(20,), (30, 15), (20, 20)]`
* Regularisation parameter `alpha`: e.g. `[0.01, 0.1, 1.0]`
* Initial learning rate `learning_rate_init`: e.g. `[0.01, 0.001]`<br>
`B` Train the optimal MLP model and evaluate its performance on train and test data using `eval_regression_perf`.

In [ ]:
# ADD HERE YOUR ANSWER TO EXERCISE 5



#### **3.5 Comparing and reflecting on the model performances and their outcomes**

In socio-technical decision analysis, evaluating multiple model families allows researchers to cross-verify findings (the "wisdom of crowds" principle). When diverse algorithms yield concordant patterns, we gain confidence in our empirical conclusions.

Let's pull together the test-set predictions of all 3 models in scatter plots:

In [ ]:
# Scatter plots: True versus Predicted Attractiveness on Test Data
fig, ax = plt.subplots(1, 3, figsize=(18, 4.5), sharex=True, sharey=True)

ax[0].scatter(Y_test, regr.predict(X_test), marker='.', s=35, alpha=0.4, c='#ff7f0e',
              label=f'Linear Reg | R2 = {regr.score(X_test, Y_test):.3f}')
ax[1].scatter(Y_test, rf.predict(X_test), marker='.', s=35, alpha=0.4, c='#2ca02c',
              label=f'Random Forest | R2 = {rf.score(X_test, Y_test):.3f}')
ax[2].scatter(Y_test, best_mlp.predict(X_test), marker='.', s=35, alpha=0.4, c='#d62728',
              label=f'MLP | R2 = {best_mlp.score(X_test, Y_test):.3f}')

for i in range(3):
    ax[i].plot([-3.5, 3.5], [-3.5, 3.5], 'k--', alpha=0.6, label='Ideal (y=x)')
    ax[i].set_xlabel('True Attractiveness (Standardized)')
    ax[i].set_ylabel('Predicted Attractiveness')
    ax[i].set_xlim(-3.5, 3.5)
    ax[i].set_ylim(-3.5, 3.5)
    ax[i].legend(loc='upper left')
    ax[i].grid(True, linestyle=':', alpha=0.6)

ax[0].set_title('Linear Regression')
ax[1].set_title('Random Forest')
ax[2].set_title('Multilayer Perceptron (MLP)')
plt.tight_layout()
plt.show()

### <span style="color:skyblue">Exercise 6: Comparison of predictions and socio-technical reflection</span>
`A` What differences do you observe between the predictions across the three ML models?<br>
`B` Notice how predictions tend to flatten slightly at the extreme ends. Why do regression models exhibit this "regression to the mean" behavior?<br>
`C` Based on the comparative performance, training complexity, and interpretability, which model would you recommend for municipal urban planners in Rotterdam? Explain your reasoning.<br>
`D` What did we gain in terms of socio-technical explainability by using semantic segmentation proportions instead of black-box image embeddings?<br>
`E` **Critical Reflection:** Suppose a municipal council proposes allocating public funding for neighborhood revitalization solely based on an automated semantic segmentation model. What are the key ethical, technical, and socio-technical risks of this automated decision system?

In [ ]:
# ADD HERE YOUR ANSWER TO EXERCISE 6


Another revealing way to compare models is through **Kernel Density Estimation (KDE)** plots, which illustrate how closely each model reproduces the true distribution and variance of the target variable:

In [ ]:
# Kernel Density Plot comparing True vs Predicted distributions
fig, ax = plt.subplots(figsize=(9, 5))

sns.kdeplot(data=Y_test, ax=ax, fill=True, alpha=0.25, color='black', label='True Test Distribution')
sns.kdeplot(data=regr.predict(X_test), ax=ax, fill=True, alpha=0.2, color='#ff7f0e',
            label=f'Linear Reg (R2={regr.score(X_test, Y_test):.3f})')
sns.kdeplot(data=rf.predict(X_test), ax=ax, fill=True, alpha=0.2, color='#2ca02c',
            label=f'Random Forest (R2={rf.score(X_test, Y_test):.3f})')
sns.kdeplot(data=best_mlp.predict(X_test), ax=ax, fill=True, alpha=0.2, color='#d62728',
            label=f'MLP (R2={best_mlp.score(X_test, Y_test):.3f})')

ax.set_title('Density Distributions: True vs. Predicted Attractiveness')
ax.set_xlabel('Standardized Attractiveness Score')
ax.set_xlim(-3.5, 3.5)
ax.legend()
plt.tight_layout()
plt.show()